##### M5 Retail Demand Forecasting
##### Dataset links
- [M5 Forecasting – Accuracy](https://www.kaggle.com/competitions/m5-forecasting-accuracy)
- [M5 data files](https://www.kaggle.com/competitions/m5-forecasting-accuracy/data)

We'll download the dataset and keep the files in `data/raw`.

##### Data Exploration
The M5 dataset records daily retail sales for products across stores. We'll use past information to predict demand in a future period, then compare predictions with actual sales.

First, we'll understand the data, look for demand patterns and check whether calendar events, SNAP indicators and prices are useful candidates for forecasting. We'll start with `sales_train_validation.csv`, `calendar.csv` and `sell_prices.csv`. The evaluation sales file overlaps with the validation history, so we won't treat the two files as separate datasets.

##### Install the project dependencies
We install dependencies like numpy, etc.

```bash
pip install -r requirements.txt
```

This installs the packages used for data handling and charts, plus the forecasting and evaluation libraries we'll need later. Make sure your notebook is using the same Python environment where you installed them. Modules like `pathlib`, `re` and `warnings` come with Python, so they don't need to be installed separately.

##### Imports and setup

We'll use pandas and NumPy for the data, Matplotlib for charts, and standard Python tools for paths and reproducibility. Since M5 is large, we'll keep memory use in mind. The path helper is designed to find `data/raw` whether this notebook sits in the project root or inside `notebooks/`.

In [ ]:
from pathlib import Path
import re
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Keep the outputs consistent and readable when we rerun the notebook.
randomSeed = 42
np.random.seed(randomSeed)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

warnings.filterwarnings("default")


# Look up from the current folder so the notebook still works if it lives in notebooks/.
def locateRawDataDir(startPath: Path | None = None, maxLevels: int = 5) -> tuple[Path, Path]:
    """Find data/raw even when the notebook is inside a subfolder."""
    # Start from the notebook working directory unless a path was supplied.
    searchStartPath = (startPath or Path.cwd()).resolve()
    parentPaths = [searchStartPath, *list(searchStartPath.parents)[:maxLevels]]

    checkedPaths = []
    # Check the current folder and a few parents for the project data folder.
    for parentPath in parentPaths:
        candidatePath = parentPath / "data" / "raw"
        checkedPaths.append(candidatePath)
        if candidatePath.is_dir():
            return candidatePath, parentPath

    checkedText = "\n".join(f"- {checkedPath}" for checkedPath in checkedPaths)
    raise FileNotFoundError(
        "Could not locate the project's data/raw directory.\n"
        "Place this notebook in the project root or a subfolder (for example, notebooks/), "
        "or start Jupyter with the project root as its working directory.\n"
        f"Directories checked:\n{checkedText}"
    )


rawDataDir, projectRoot = locateRawDataDir()
reportsDir = projectRoot / "reports"
tablesDir = reportsDir / "tables"
figuresDir = reportsDir / "figures"

# These folders are for our outputs. The original files in data/raw stay untouched.
tablesDir.mkdir(parents=True, exist_ok=True)
figuresDir.mkdir(parents=True, exist_ok=True)

print(f"Python version : {sys.version.split()[0]}")
print(f"Pandas version : {pd.__version__}")
print(f"NumPy version  : {np.__version__}")
print(f"Project root   : {projectRoot}")
print(f"Raw data folder: {rawDataDir}")

##### Load the data
Let's list the available files and their sizes first. This confirms what is actually downloaded before we try to load it.

In [ ]:
fileInventory = pd.DataFrame(
    [
        {
            "file_name": filePath.name,
            "size_mb": filePath.stat().st_size / (1024 ** 2),
            "extension": filePath.suffix.lower(),
        }
        for filePath in sorted(rawDataDir.iterdir())
        if filePath.is_file()
    ]
)

if fileInventory.empty:
    raise FileNotFoundError(f"No files were found in {rawDataDir}")

display(fileInventory)

# These are the three files we need to get started.
salesFilePath = rawDataDir / "sales_train_validation.csv"
calendarFilePath = rawDataDir / "calendar.csv"
pricesFilePath = rawDataDir / "sell_prices.csv"

requiredFiles = [salesFilePath, calendarFilePath, pricesFilePath]
missingFiles = [filePath.name for filePath in requiredFiles if not filePath.is_file()]

if missingFiles:
    raise FileNotFoundError(
        "The following required M5 files are missing from data/raw: "
        + ", ".join(missingFiles)
        + ". Download the files from the official Kaggle competition data page."
    )

print("All required files are present.")
print("Optional files detected:")
for optionalName in ["sales_train_evaluation.csv", "sample_submission.csv"]:
    optionalPath = rawDataDir / optionalName
    print(f"- {optionalName}: {'available' if optionalPath.is_file() else 'not found'}")

##### Check the sales columns before loading everything

The sales file is in wide format: each row is a product-store combination, and each `d_` column represents a day. We'll detect those columns from the header and use compact types to reduce memory use. The chosen sales type assumes non-negative counts within its range, so we'll check that assumption.

In [ ]:
salesHeader = pd.read_csv(salesFilePath, nrows=0)
salesColumnNames = salesHeader.columns.tolist()

dayColumns = [
    columnName for columnName in salesColumnNames
    if re.fullmatch(r"d_\d+", str(columnName))
]
identifierColumns = [
    columnName for columnName in ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
    if columnName in salesColumnNames
]

if not dayColumns:
    raise ValueError(
        "No daily sales columns matching the expected pattern d_<number> were found. "
        "Please check that the correct M5 sales file was downloaded."
    )

print(f"Number of identifier/metadata columns: {len(identifierColumns)}")
print(f"Number of daily sales columns: {len(dayColumns)}")
print("First daily columns:", dayColumns[:5])
print("Last daily columns :", dayColumns[-5:])

# M5 has a lot of daily columns, so compact types help keep memory use under control.
salesDataTypes = {columnName: "category" for columnName in identifierColumns}
salesDataTypes.update({columnName: "uint16" for columnName in dayColumns})

# Load the wide sales table with compact types to keep memory use reasonable.
salesData = pd.read_csv(
    salesFilePath,
    dtype=salesDataTypes,
    low_memory=False,
)

# Parse dates properly, and store repeated labels compactly where we can.
calendarCategoryColumns = [
    "d", "weekday", "event_name_1", "event_type_1",
    "event_name_2", "event_type_2",
]
calendarHeader = pd.read_csv(calendarFilePath, nrows=0)
calendarDataTypes = {
    columnName: "category"
    for columnName in calendarCategoryColumns
    if columnName in calendarHeader.columns
}
calendarData = pd.read_csv(
    calendarFilePath,
    dtype=calendarDataTypes,
    parse_dates=["date"],
    low_memory=False,
)

# Store and item IDs repeat a lot in the price file, so category types save memory here too.
priceHeader = pd.read_csv(pricesFilePath, nrows=0)
priceDataTypes = {
    columnName: "category"
    for columnName in ["store_id", "item_id"]
    if columnName in priceHeader.columns
}
if "wm_yr_wk" in priceHeader.columns:
    priceDataTypes["wm_yr_wk"] = "int32"
if "sell_price" in priceHeader.columns:
    priceDataTypes["sell_price"] = "float32"

priceData = pd.read_csv(
    pricesFilePath,
    dtype=priceDataTypes,
    low_memory=False,
)

print("Data loaded successfully.")

In [ ]:
# Take a quick look at the size and shape of each file before digging in.
datasetOverview = pd.DataFrame(
    [
        {
            "dataset": "sales",
            "rows": len(salesData),
            "columns": salesData.shape[1],
            "memory_mb": salesData.memory_usage(deep=True).sum() / (1024 ** 2),
        },
        {
            "dataset": "calendar",
            "rows": len(calendarData),
            "columns": calendarData.shape[1],
            "memory_mb": calendarData.memory_usage(deep=True).sum() / (1024 ** 2),
        },
        {
            "dataset": "prices",
            "rows": len(priceData),
            "columns": priceData.shape[1],
            "memory_mb": priceData.memory_usage(deep=True).sum() / (1024 ** 2),
        },
    ]
)

display(datasetOverview)

print("Sales data sample:")
display(salesData.iloc[:5, :min(12, salesData.shape[1])])

print("Calendar data sample:")
display(calendarData.head())

print("Price data sample:")
display(priceData.head())

print("Sales metadata data types:")
display(salesData[identifierColumns].dtypes.astype(str).rename("dtype").to_frame())

print("Calendar data types:")
display(calendarData.dtypes.astype(str).rename("dtype").to_frame())

print("Price data types:")
display(priceData.dtypes.astype(str).rename("dtype").to_frame())

#### Checking the quality of the dataset

In this section we'll check for missing values, duplicate keys, date mappings, unexpected sales or prices, zero-sales observations, and whether product/store identifiers line up across files. A missing value doesn't always mean a bad record: blank event fields can be normal, and zero sales are worth investigating rather than automatically removing.

In [ ]:
def missingValueSummary(dataFrame: pd.DataFrame, datasetName: str) -> pd.DataFrame:
    """Show only the columns where we actually have missing values."""
    missingCounts = dataFrame.isna().sum()
    missingCounts = missingCounts[missingCounts > 0]

    if missingCounts.empty:
        return pd.DataFrame(columns=["dataset", "column", "missing_count", "missing_percent"])

    return pd.DataFrame({
        "dataset": datasetName,
        "column": missingCounts.index,
        "missing_count": missingCounts.values,
        "missing_percent": [
            100 * valueCount / len(dataFrame) if len(dataFrame) else np.nan
            for valueCount in missingCounts.values
        ],
    })


# Keep this report focused on columns that actually have missing values.
missingReport = pd.concat(
    [
        missingValueSummary(salesData, "sales"),
        missingValueSummary(calendarData, "calendar"),
        missingValueSummary(priceData, "prices"),
    ],
    ignore_index=True,
)

if missingReport.empty:
    print("No missing values were found in the loaded datasets.")
else:
    print("Columns containing missing values:")
    display(missingReport.sort_values(["dataset", "missing_count"], ascending=[True, False]))

In [ ]:
# A few checks to make sure the files are behaving as expected.

# Each sales ID should point to one product at one store.
duplicateSalesIds = int(salesData.duplicated(subset=["id"]).sum()) if "id" in salesData.columns else None
productStoreKey = [columnName for columnName in ["item_id", "store_id"] if columnName in salesData.columns]
duplicateProductStoreKeys = (
    int(salesData.duplicated(subset=productStoreKey).sum())
    if len(productStoreKey) == 2 else None
)

# Every M5 day ID should appear once in the calendar.
duplicateCalendarDayIds = (
    int(calendarData.duplicated(subset=["d"]).sum()) if "d" in calendarData.columns else None
)
duplicateCalendarDates = (
    int(calendarData.duplicated(subset=["date"]).sum()) if "date" in calendarData.columns else None
)

# There should be one price record per store, item and week.
priceKey = [columnName for columnName in ["store_id", "item_id", "wm_yr_wk"] if columnName in priceData.columns]
duplicatePriceKeys = (
    int(priceData.duplicated(subset=priceKey).sum())
    if len(priceKey) == 3 else None
)

# Make sure every sales-day column has a matching calendar entry.
calendarDayIds = set(calendarData["d"].astype(str)) if "d" in calendarData.columns else set()
salesDaysMissingFromCalendar = [day for day in dayColumns if day not in calendarDayIds]

# Zero sales are worth counting; negative unit sales would need a closer look.
salesValues = salesData[dayColumns]
totalSalesObservations = int(salesValues.size)
zeroSalesObservations = int(salesValues.eq(0).to_numpy().sum())
negativeSalesObservations = int(salesValues.lt(0).to_numpy().sum())
zeroSalesPercent = (
    100 * zeroSalesObservations / totalSalesObservations
    if totalSalesObservations else np.nan
)
allZeroSeriesCount = int(salesValues.sum(axis=1).eq(0).sum())

# Check the prices for missing or non-positive values before we use them.
missingPriceValues = int(priceData["sell_price"].isna().sum()) if "sell_price" in priceData.columns else None
nonpositivePriceValues = (
    int(priceData["sell_price"].le(0).sum())
    if "sell_price" in priceData.columns else None
)

# Check that the products and stores in the sales file also appear in the price file.
salesItemIds = set(salesData["item_id"].astype(str)) if "item_id" in salesData.columns else set()
priceItemIds = set(priceData["item_id"].astype(str)) if "item_id" in priceData.columns else set()
salesStoreIds = set(salesData["store_id"].astype(str)) if "store_id" in salesData.columns else set()
priceStoreIds = set(priceData["store_id"].astype(str)) if "store_id" in priceData.columns else set()

qualityChecks = [
    ("Sales rows", len(salesData), "Count of product-store series in the sales file"),
    ("Daily sales columns", len(dayColumns), "Number of historical daily observations per series"),
    ("Unique item IDs in sales", len(salesItemIds), "Distinct product identifiers"),
    ("Unique store IDs in sales", len(salesStoreIds), "Distinct store identifiers"),
    ("Duplicate sales IDs", duplicateSalesIds, "Expected to be zero"),
    ("Duplicate item-store keys", duplicateProductStoreKeys, "Expected to be zero"),
    ("Duplicate calendar day IDs", duplicateCalendarDayIds, "Expected to be zero"),
    ("Duplicate calendar dates", duplicateCalendarDates, "Inspect if non-zero"),
    ("Sales day columns without calendar mapping", len(salesDaysMissingFromCalendar), "Expected to be zero"),
    ("Duplicate store-item-week price keys", duplicatePriceKeys, "Expected to be zero"),
    ("Missing sell-price values", missingPriceValues, "Inspect before any price-based modelling"),
    ("Non-positive sell-price values", nonpositivePriceValues, "Inspect before any price-based modelling"),
    ("Sales observations with negative values", negativeSalesObservations, "Expected to be zero for unit sales"),
    ("Zero-sales observations (%)", round(zeroSalesPercent, 3), "Zero cells as a percentage of all daily sales cells"),
    ("Series with zero sales throughout", allZeroSeriesCount, "Investigate if any are present"),
    ("Sales item IDs absent from prices", len(salesItemIds - priceItemIds), "Check before price feature joins"),
    ("Sales store IDs absent from prices", len(salesStoreIds - priceStoreIds), "Check before price feature joins"),
]

qualityReport = pd.DataFrame(
    qualityChecks,
    columns=["check", "result", "interpretation"]
)

display(qualityReport)

if salesDaysMissingFromCalendar:
    print("Sales day columns without calendar mappings:", salesDaysMissingFromCalendar[:20])
else:
    print("All sales day columns have matching calendar day identifiers.")

In [ ]:
# Check that the day IDs are in order and none are missing from the sequence.
dayNumbers = [int(columnName.split("_", maxsplit=1)[1]) for columnName in dayColumns]
expectedDayNumbers = list(range(min(dayNumbers), max(dayNumbers) + 1)) if dayNumbers else []
missingDayNumbers = sorted(set(expectedDayNumbers) - set(dayNumbers))

# Translate the first and last day IDs into dates we can read.
if "d" in calendarData.columns and "date" in calendarData.columns:
    calendarDateMap = calendarData.assign(d=calendarData["d"].astype(str)).set_index("d")["date"]
    mappedDates = calendarDateMap.reindex([dayColumns[0], dayColumns[-1]])
    firstSalesDate = mappedDates.iloc[0]
    lastSalesDate = mappedDates.iloc[-1]
else:
    firstSalesDate = pd.NaT
    lastSalesDate = pd.NaT

print(f"Day identifiers form a continuous sequence: {len(missingDayNumbers) == 0}")
if missingDayNumbers:
    print("Missing day numbers:", missingDayNumbers[:20])
print(f"First sales day: {dayColumns[0]} -> {firstSalesDate}")
print(f"Last sales day : {dayColumns[-1]} -> {lastSalesDate}")

# Show the useful schema summary without dumping every daily sales column.
schemaSummary = pd.DataFrame([
    {
        "dataset": "sales",
        "rows": salesData.shape[0],
        "columns": salesData.shape[1],
        "identifier_columns": ", ".join(identifierColumns),
        "daily_columns": len(dayColumns),
        "memory_mb": round(salesData.memory_usage(deep=True).sum() / (1024 ** 2), 2),
    },
    {
        "dataset": "calendar",
        "rows": calendarData.shape[0],
        "columns": calendarData.shape[1],
        "identifier_columns": "d, date",
        "daily_columns": np.nan,
        "memory_mb": round(calendarData.memory_usage(deep=True).sum() / (1024 ** 2), 2),
    },
    {
        "dataset": "prices",
        "rows": priceData.shape[0],
        "columns": priceData.shape[1],
        "identifier_columns": "store_id, item_id, wm_yr_wk",
        "daily_columns": np.nan,
        "memory_mb": round(priceData.memory_usage(deep=True).sum() / (1024 ** 2), 2),
    },
])

display(schemaSummary)

print("The next section performs exploratory data analysis.")

##### Exploratory data analysis (EDA)

We'll explore overall sales, store and product differences, zero-sales days, volatility, seasonality, events, SNAP indicators and prices. For each analysis, we'll record what we see and how it might affect the forecasting approach. These patterns show associations, not necessarily causes.

Charts will be saved in `reports/figures/` and summary tables in `reports/tables/`. We'll avoid reshaping the full dataset unless an analysis needs it.

##### Set up the EDA data

A *series* is one product sold at one store. We'll calculate most summaries directly from the wide sales table to save memory, and reshape only a manageable sample for price analysis.

In [ ]:
# A couple of helpers keep our charts and tables in one place.
def saveCurrentFigure(fileName: str) -> None:
    """Save the chart, show it in the notebook, and close it afterwards."""
    figure = plt.gcf()
    figure.tight_layout()
    outputPath = figuresDir / fileName
    figure.savefig(outputPath, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(figure)
    print(f"Saved figure: {outputPath.relative_to(projectRoot)}")


def saveTable(dataFrame: pd.DataFrame, fileName: str, includeIndex: bool = False) -> None:
    """Save a summary table without touching the original data files."""
    outputPath = tablesDir / fileName
    dataFrame.to_csv(outputPath, index=includeIndex)
    print(f"Saved table: {outputPath.relative_to(projectRoot)}")


# Work on a copy of the calendar so the loaded data stays unchanged.
calendarForEda = calendarData.copy()
calendarForEda["d"] = calendarForEda["d"].astype(str)
calendarForEda = calendarForEda.sort_values("date").reset_index(drop=True)

# Check the sales day IDs against the calendar before using the dates.
calendarDayMap = calendarForEda.set_index("d")
missingCalendarDays = [day for day in dayColumns if day not in calendarDayMap.index]
if missingCalendarDays:
    raise ValueError(
        f"Cannot continue EDA: {len(missingCalendarDays)} sales-day columns do not have calendar records. "
        f"Examples: {missingCalendarDays[:5]}"
    )

# Add up sales across products and stores to get one total for each day.
dailySales = pd.DataFrame({
    "d": dayColumns,
    "total_units_sold": salesData[dayColumns].sum(axis=0).to_numpy(dtype=np.int64),
})
dailySales = dailySales.merge(
    calendarForEda,
    on="d",
    how="left",
    validate="one_to_one",
    suffixes=("", "_calendar"),
)
dailySales = dailySales.sort_values("date").reset_index(drop=True)
dailySales["month_num"] = dailySales["date"].dt.month
dailySales["year_num"] = dailySales["date"].dt.year
dailySales["day_of_week_num"] = dailySales["date"].dt.dayofweek

# Summarize each product-store series directly from the wide table; no giant reshape needed.
dailyMatrix = salesData[dayColumns]
seriesSummary = salesData[identifierColumns].copy()
seriesSummary["total_units"] = dailyMatrix.sum(axis=1).to_numpy(dtype=np.int64)
seriesSummary["mean_daily_units"] = dailyMatrix.mean(axis=1).to_numpy(dtype=np.float64)
seriesSummary["std_daily_units"] = dailyMatrix.std(axis=1, ddof=0).to_numpy(dtype=np.float64)
seriesSummary["nonzero_days"] = dailyMatrix.gt(0).sum(axis=1).to_numpy(dtype=np.int32)
seriesSummary["zero_day_share"] = 1.0 - seriesSummary["nonzero_days"] / len(dayColumns)
seriesSummary["mean_nonzero_units"] = np.divide(
    seriesSummary["total_units"].to_numpy(dtype=np.float64),
    seriesSummary["nonzero_days"].to_numpy(dtype=np.float64),
    out=np.full(len(seriesSummary), np.nan, dtype=np.float64),
    where=seriesSummary["nonzero_days"].to_numpy() > 0,
)
seriesSummary["average_demand_interval"] = np.divide(
    len(dayColumns),
    seriesSummary["nonzero_days"].to_numpy(dtype=np.float64),
    out=np.full(len(seriesSummary), np.nan, dtype=np.float64),
    where=seriesSummary["nonzero_days"].to_numpy() > 0,
)
seriesSummary["cv_all_days"] = np.divide(
    seriesSummary["std_daily_units"].to_numpy(dtype=np.float64),
    seriesSummary["mean_daily_units"].to_numpy(dtype=np.float64),
    out=np.full(len(seriesSummary), np.nan, dtype=np.float64),
    where=seriesSummary["mean_daily_units"].to_numpy() > 0,
)

# Calculate variability on positive-sales days separately; zeros affect the all-days CV above.
positiveDayCv = np.full(len(salesData), np.nan, dtype=np.float64)
chunkSize = 512
# Do this in chunks so we do not make one enormous float array in memory.
for chunkStart in range(0, len(salesData), chunkSize):
    chunkEnd = min(chunkStart + chunkSize, len(salesData))
    salesChunkValues = salesData.iloc[chunkStart:chunkEnd][dayColumns].to_numpy(dtype=np.float64, copy=True)
    # Only positive-demand days count in this second CV calculation.
    positiveDayCounts = (salesChunkValues > 0).sum(axis=1)
    positiveSalesSums = salesChunkValues.sum(axis=1)
    squaredSums = np.einsum("ij,ij->i", salesChunkValues, salesChunkValues)
    meanPositiveSales = np.divide(
        positiveSalesSums, positiveDayCounts,
        out=np.full(len(salesChunkValues), np.nan, dtype=np.float64),
        where=positiveDayCounts > 0,
    )
    positiveSalesVariances = np.divide(
        squaredSums, positiveDayCounts,
        out=np.full(len(salesChunkValues), np.nan, dtype=np.float64),
        where=positiveDayCounts > 0,
    ) - meanPositiveSales ** 2
    positiveSalesStdDev = np.sqrt(np.maximum(positiveSalesVariances, 0.0))
    # A CV is only defined here when the series has positive sales and a positive mean.
    positiveDayCv[chunkStart:chunkEnd] = np.divide(
        positiveSalesStdDev, meanPositiveSales,
        out=np.full(len(salesChunkValues), np.nan, dtype=np.float64),
        where=(positiveDayCounts > 0) & (meanPositiveSales > 0),
    )
seriesSummary["cv_nonzero_demand"] = positiveDayCv

print(f"Sales series: {len(seriesSummary):,}")
print(f"Sales days per series: {len(dayColumns):,}")
print(f"Date range: {dailySales['date'].min().date()} to {dailySales['date'].max().date()}")
print(f"Total units recorded in the sales history: {dailySales['total_units_sold'].sum():,}")
display(seriesSummary.head())

saveTable(dailySales[["d", "date", "total_units_sold"]], "daily_total_sales.csv")
saveTable(seriesSummary, "series_level_eda.csv")

##### What do sales look like over time?

We'll plot total daily sales, then weekly and monthly totals to see the broader trend. Overall totals can hide differences between products and stores, so we'll break those down next.

In [ ]:
# The rolling averages make the longer pattern easier to see through day-to-day noise.
dailySales["rolling_mean_7d"] = dailySales["total_units_sold"].rolling(7, min_periods=1).mean()
dailySales["rolling_mean_28d"] = dailySales["total_units_sold"].rolling(28, min_periods=1).mean()

figure, axis = plt.subplots(figsize=(14, 6))
axis.plot(dailySales["date"], dailySales["total_units_sold"], linewidth=0.7, alpha=0.55, label="Daily total sales")
axis.plot(dailySales["date"], dailySales["rolling_mean_7d"], linewidth=1.3, label="7-day rolling mean")
axis.plot(dailySales["date"], dailySales["rolling_mean_28d"], linewidth=1.8, label="28-day rolling mean")
axis.set_title("Total daily unit sales across all stores and products")
axis.set_xlabel("Date")
axis.set_ylabel("Units sold")
axis.legend()
saveCurrentFigure("eda_total_daily_sales.png")

weeklySales = (
    dailySales.set_index("date")["total_units_sold"]
    .resample("W-SAT").sum().rename("weekly_units_sold").reset_index()
)
monthlySales = (
    dailySales.assign(month_start=dailySales["date"].dt.to_period("M").dt.to_timestamp())
    .groupby("month_start", as_index=False)["total_units_sold"].sum()
    .rename(columns={"total_units_sold": "monthly_units_sold"})
)

figure, axis = plt.subplots(figsize=(14, 5))
axis.plot(weeklySales["date"], weeklySales["weekly_units_sold"], linewidth=1.0)
axis.set_title("Weekly unit sales across all stores and products")
axis.set_xlabel("Week ending")
axis.set_ylabel("Units sold per week")
saveCurrentFigure("eda_weekly_total_sales.png")

figure, axis = plt.subplots(figsize=(14, 5))
axis.plot(monthlySales["month_start"], monthlySales["monthly_units_sold"], marker="o", markersize=2.5, linewidth=1.1)
axis.set_title("Monthly unit sales across all stores and products")
axis.set_xlabel("Month")
axis.set_ylabel("Units sold per month")
saveCurrentFigure("eda_monthly_total_sales.png")

saveTable(weeklySales, "weekly_total_sales.csv")
saveTable(monthlySales, "monthly_total_sales.csv")
display(dailySales[["date", "total_units_sold", "rolling_mean_7d", "rolling_mean_28d"]].describe().T)

##### Which stores and product groups account for sales?

We'll compare sales by state, store, category and department. Since total sales also depend on group size, we'll use per-series summaries where they make comparisons fairer.

In [ ]:
# We already calculated totals per product-store series, so use those summaries here.
# Start with the bigger groups, then work down to stores and product groups.
if "state_id" in seriesSummary.columns:
    stateTotals = (
        seriesSummary.groupby("state_id", observed=True, as_index=False)["total_units"]
        .sum().sort_values("total_units", ascending=False)
    )
    display(stateTotals)
    figure, axis = plt.subplots(figsize=(8, 5))
    axis.bar(stateTotals["state_id"].astype(str), stateTotals["total_units"])
    axis.set_title("Total recorded unit sales by state")
    axis.set_xlabel("State")
    axis.set_ylabel("Units sold across the observed period")
    saveCurrentFigure("eda_sales_by_state.png")
    saveTable(stateTotals, "sales_by_state.csv")

if "store_id" in seriesSummary.columns:
    storeTotals = (
        seriesSummary.groupby([columnName for columnName in ["state_id", "store_id"] if columnName in seriesSummary.columns],
                               observed=True, as_index=False)["total_units"]
        .sum().sort_values("total_units", ascending=False)
    )
    topStores = storeTotals.head(15).copy()
    display(topStores)
    figure, axis = plt.subplots(figsize=(11, 6))
    labels = topStores["store_id"].astype(str)
    axis.barh(labels[::-1], topStores["total_units"].to_numpy()[::-1])
    axis.set_title("15 stores with the highest total recorded unit sales")
    axis.set_xlabel("Units sold across the observed period")
    axis.set_ylabel("Store")
    saveCurrentFigure("eda_top_stores_by_sales.png")
    saveTable(storeTotals, "sales_by_store.csv")

for groupColumn, outputName, chartTitle in [
    ("cat_id", "sales_by_category.csv", "Total recorded unit sales by product category"),
    ("dept_id", "sales_by_department.csv", "Total recorded unit sales by department"),
]:
    if groupColumn in seriesSummary.columns:
        groupTotals = (
            seriesSummary.groupby(groupColumn, observed=True, as_index=False)["total_units"]
            .sum().sort_values("total_units", ascending=False)
        )
        display(groupTotals)
        figure, axis = plt.subplots(figsize=(10, 5))
        axis.bar(groupTotals[groupColumn].astype(str), groupTotals["total_units"])
        axis.set_title(chartTitle)
        axis.set_xlabel(groupColumn.replace("_", " ").title())
        axis.set_ylabel("Units sold across the observed period")
        axis.tick_params(axis="x", rotation=30)
        saveCurrentFigure(f"eda_{groupColumn}_sales.png")
        saveTable(groupTotals, outputName)

# Average series-level demand gives us a fairer comparison than category totals alone.
if "cat_id" in seriesSummary.columns:
    categoryStats = (
        seriesSummary.groupby("cat_id", observed=True)
        .agg(
            series_count=("id", "size"),
            median_mean_daily_units=("mean_daily_units", "median"),
            median_total_units=("total_units", "median"),
            mean_zero_day_share=("zero_day_share", "mean"),
        ).reset_index()
    )
    display(categoryStats)
    saveTable(categoryStats, "category_series_summary.csv")

##### How common are zero-sales days?

We'll measure how often each product-store series records zero sales. Frequent zeros can make forecasting harder, but they don't tell us whether an item was unavailable, newly introduced or simply had no sales.

In [ ]:
# Count zeros across every observed product-store day, then look at how zeros vary by series.
zeroObservations = int((salesData[dayColumns] == 0).to_numpy().sum())
totalObservations = int(len(salesData) * len(dayColumns))
zeroShareOverall = zeroObservations / totalObservations if totalObservations else np.nan

print(f"All daily series observations: {totalObservations:,}")
print(f"Zero-sales observations: {zeroObservations:,} ({zeroShareOverall:.2%})")
print(f"Series with no recorded sales throughout the history: {(seriesSummary['total_units'] == 0).sum():,}")

sparsityThresholds = pd.DataFrame({
    "condition": [
        "At least 25% of days have zero sales",
        "At least 50% of days have zero sales",
        "At least 75% of days have zero sales",
        "At least 90% of days have zero sales",
        "No recorded sales over the full history",
    ],
    "series_count": [
        int((seriesSummary["zero_day_share"] >= 0.25).sum()),
        int((seriesSummary["zero_day_share"] >= 0.50).sum()),
        int((seriesSummary["zero_day_share"] >= 0.75).sum()),
        int((seriesSummary["zero_day_share"] >= 0.90).sum()),
        int((seriesSummary["total_units"] == 0).sum()),
    ],
})
sparsityThresholds["percent_of_series"] = 100 * sparsityThresholds["series_count"] / len(seriesSummary)
display(sparsityThresholds)
saveTable(sparsityThresholds, "demand_sparsity_thresholds.csv")

figure, axis = plt.subplots(figsize=(10, 5))
axis.hist(seriesSummary["zero_day_share"].dropna(), bins=30, edgecolor="black", linewidth=0.3)
axis.set_title("Distribution of zero-sales-day proportion across product-store series")
axis.set_xlabel("Share of observed days with zero sales")
axis.set_ylabel("Number of product-store series")
saveCurrentFigure("eda_zero_sales_share_distribution.png")

if "cat_id" in seriesSummary.columns:
    sparsityByCategory = (
        seriesSummary.groupby("cat_id", observed=True)
        .agg(
            series_count=("id", "size"),
            mean_zero_day_share=("zero_day_share", "mean"),
            median_zero_day_share=("zero_day_share", "median"),
            series_at_least_half_zero=("zero_day_share", lambda dailyValues: int((dailyValues >= 0.5).sum())),
        ).reset_index()
    )
    sparsityByCategory["percent_series_at_least_half_zero"] = (
        100 * sparsityByCategory["series_at_least_half_zero"] / sparsityByCategory["series_count"]
    )
    display(sparsityByCategory)
    figure, axis = plt.subplots(figsize=(8, 5))
    axis.bar(sparsityByCategory["cat_id"].astype(str), sparsityByCategory["mean_zero_day_share"])
    axis.set_title("Average zero-sales-day share by product category")
    axis.set_xlabel("Product category")
    axis.set_ylabel("Mean zero-sales-day share")
    saveCurrentFigure("eda_zero_sales_share_by_category.png")
    saveTable(sparsityByCategory, "sparsity_by_category.csv")

##### How variable is demand?

We'll measure variability using the coefficient of variation (CV) across all days and on positive-sales days only. We'll also calculate the average demand interval (ADI), which describes how often sales occur. CV can be unstable when mean demand is very low, so we'll interpret it alongside zero-sales share and ADI.

In [ ]:
volatilitySummary = seriesSummary[
    ["id", "cat_id", "dept_id", "store_id", "total_units", "mean_daily_units",
     "std_daily_units", "zero_day_share", "average_demand_interval",
     "cv_all_days", "cv_nonzero_demand"]
].copy()

# Keep infinite CV values out of the summaries; they can occur when average demand is zero.
validCvAll = volatilitySummary["cv_all_days"].replace([np.inf, -np.inf], np.nan).dropna()
validCvPositive = volatilitySummary["cv_nonzero_demand"].replace([np.inf, -np.inf], np.nan).dropna()

print("CV across all days — summary:")
display(validCvAll.describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.95, 0.99]).to_frame("cv_all_days"))
print("CV of positive-demand days — summary:")
display(validCvPositive.describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.95, 0.99]).to_frame("cv_nonzero_demand"))

# A few extreme values can flatten the chart, so cap the displayed tail for readability only.
if not validCvAll.empty:
    upperCv = validCvAll.quantile(0.99)
    figure, axis = plt.subplots(figsize=(10, 5))
    axis.hist(validCvAll.clip(upper=upperCv), bins=50, edgecolor="black", linewidth=0.25)
    axis.set_title("Distribution of demand CV across all observed days (visualization capped at p99)")
    axis.set_xlabel("CV across all days")
    axis.set_ylabel("Number of series")
    saveCurrentFigure("eda_cv_all_days_distribution.png")

if not validCvPositive.empty:
    upperCvPositive = validCvPositive.quantile(0.99)
    figure, axis = plt.subplots(figsize=(10, 5))
    axis.hist(validCvPositive.clip(upper=upperCvPositive), bins=50, edgecolor="black", linewidth=0.25)
    axis.set_title("Distribution of CV on positive-demand days (visualization capped at p99)")
    axis.set_xlabel("CV of positive-demand quantities")
    axis.set_ylabel("Number of series")
    saveCurrentFigure("eda_cv_positive_days_distribution.png")

# Use a sample for this plot; we do not need every series to see the broad pattern.
scatterData = volatilitySummary.dropna(subset=["zero_day_share", "cv_all_days"]).sample(
    n=min(5000, volatilitySummary.dropna(subset=["zero_day_share", "cv_all_days"]).shape[0]),
    random_state=randomSeed,
)
figure, axis = plt.subplots(figsize=(10, 6))
axis.scatter(scatterData["zero_day_share"], scatterData["cv_all_days"], alpha=0.3, s=12)
axis.set_title("Demand sparsity versus CV across all days (sample of series)")
axis.set_xlabel("Share of days with zero sales")
axis.set_ylabel("CV across all days")
saveCurrentFigure("eda_sparsity_vs_volatility.png")

if "cat_id" in volatilitySummary.columns:
    volatilityByCategory = (
        volatilitySummary.groupby("cat_id", observed=True)
        .agg(
            series_count=("id", "size"),
            median_cv_all_days=("cv_all_days", "median"),
            median_cv_nonzero=("cv_nonzero_demand", "median"),
            median_adi=("average_demand_interval", "median"),
            median_zero_day_share=("zero_day_share", "median"),
        ).reset_index()
    )
    display(volatilityByCategory)
    saveTable(volatilityByCategory, "volatility_by_category.csv")

# These low/medium/high groups are only for exploring the data. They are relative to this dataset,
# so we will set any formal modelling thresholds using training data only.
cvValidMask = seriesSummary["cv_all_days"].notna()
seriesSummary["volatility_tercile_eda"] = pd.NA
if cvValidMask.sum() >= 3:
    ranks = seriesSummary.loc[cvValidMask, "cv_all_days"].rank(method="first", pct=True)
    seriesSummary.loc[cvValidMask, "volatility_tercile_eda"] = pd.cut(
        ranks,
        bins=[0, 1/3, 2/3, 1.0000001],
        labels=["Low", "Medium", "High"],
        include_lowest=True,
    ).astype(str)
print("Volatility terciles are exploratory only; determine thresholds from training data for formal evaluation.")

##### Do sales follow calendar patterns?

We'll compare sales by weekday, month and year. These patterns may help us decide whether calendar features or a weekly seasonal baseline are useful. We'll be careful with year-on-year comparisons when the first or last year is incomplete.

In [ ]:
# Keep weekdays in calendar order instead of sorting them alphabetically.
weekdayOrder = ["Saturday", "Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
weekdayMeans = (
    dailySales.groupby("weekday", observed=True)["total_units_sold"].mean()
    .reindex(weekdayOrder).dropna().rename("mean_daily_units").reset_index()
)
weekdayMeans["index_vs_overall_mean"] = (
    100 * weekdayMeans["mean_daily_units"] / dailySales["total_units_sold"].mean()
)

display(weekdayMeans)
figure, axis = plt.subplots(figsize=(10, 5))
axis.bar(weekdayMeans["weekday"].astype(str), weekdayMeans["mean_daily_units"])
axis.set_title("Average total daily sales by weekday")
axis.set_xlabel("Weekday")
axis.set_ylabel("Mean units sold per day")
axis.tick_params(axis="x", rotation=25)
saveCurrentFigure("eda_average_sales_by_weekday.png")
saveTable(weekdayMeans, "sales_by_weekday.csv")

monthOrder = list(range(1, 13))
monthlySeasonality = (
    dailySales.groupby("month_num", observed=True)["total_units_sold"].mean()
    .reindex(monthOrder).rename("mean_daily_units").reset_index()
)
monthlySeasonality["month_name"] = monthlySeasonality["month_num"].map(lambda monthNumber: pd.Timestamp(2000, int(monthNumber), 1).strftime("%b"))
display(monthlySeasonality)
figure, axis = plt.subplots(figsize=(10, 5))
axis.plot(monthlySeasonality["month_name"], monthlySeasonality["mean_daily_units"], marker="o")
axis.set_title("Average daily sales by calendar month")
axis.set_xlabel("Month")
axis.set_ylabel("Mean units sold per day")
saveCurrentFigure("eda_average_sales_by_month_of_year.png")
saveTable(monthlySeasonality, "sales_by_month_of_year.csv")

# Looking at each year separately helps us see whether the month pattern repeats.
monthlyByYear = (
    dailySales.assign(month_start=dailySales["date"].dt.to_period("M").dt.to_timestamp())
    .groupby(["year_num", "month_num"], observed=True)
    .agg(monthly_units=("total_units_sold", "sum"), days_observed=("date", "count"))
    .reset_index()
)
monthlyByYear["mean_daily_units"] = monthlyByYear["monthly_units"] / monthlyByYear["days_observed"]
figure, axis = plt.subplots(figsize=(12, 6))
for calendarYear, seriesGroup in monthlyByYear.groupby("year_num", observed=True):
    axis.plot(seriesGroup["month_num"], seriesGroup["mean_daily_units"], marker="o", markersize=3, label=str(calendarYear))
axis.set_title("Average daily sales by month, separated by year")
axis.set_xlabel("Month number")
axis.set_ylabel("Mean units sold per observed day")
axis.set_xticks(monthOrder)
axis.legend(title="Year", ncol=2)
saveCurrentFigure("eda_monthly_pattern_by_year.png")
saveTable(monthlyByYear, "monthly_sales_by_year.csv")

##### Events and holidays

We'll compare sales on dates with recorded events against other dates and summarize event names that occur often enough. This is descriptive only: weekday, season and trend may also explain differences. Blank event names are normal for many calendar dates.

In [ ]:
# First compare days with any recorded event, then inspect individual event names/types.
eventColumns = [columnName for columnName in ["event_name_1", "event_name_2"] if columnName in dailySales.columns]
if eventColumns:
    dailySales["has_recorded_event"] = dailySales[eventColumns].notna().any(axis=1)
    eventComparison = (
        dailySales.groupby("has_recorded_event", observed=True)
        .agg(
            number_of_days=("date", "count"),
            mean_daily_units=("total_units_sold", "mean"),
            median_daily_units=("total_units_sold", "median"),
            std_daily_units=("total_units_sold", "std"),
        ).reset_index()
    )
    eventComparison["day_type"] = eventComparison["has_recorded_event"].map({True: "Recorded event", False: "No recorded event"})
    display(eventComparison)

    figure, axis = plt.subplots(figsize=(8, 5))
    axis.bar(eventComparison["day_type"], eventComparison["mean_daily_units"])
    axis.set_title("Mean total daily sales: recorded event vs no recorded event")
    axis.set_xlabel("Calendar-day group")
    axis.set_ylabel("Mean units sold per day")
    saveCurrentFigure("eda_event_vs_non_event_sales.png")
    saveTable(eventComparison, "event_day_comparison.csv")

    eventNameFrames = []
    for eventColumn in eventColumns:
        eventRows = dailySales.loc[dailySales[eventColumn].notna(), ["date", "total_units_sold", eventColumn]].copy()
        eventRows = eventRows.rename(columns={eventColumn: "event_name"})
        eventRows["event_source_column"] = eventColumn
        eventNameFrames.append(eventRows)
    eventRecords = pd.concat(eventNameFrames, ignore_index=True) if eventNameFrames else pd.DataFrame()
    if not eventRecords.empty:
        eventNameSummary = (
            eventRecords.groupby(["event_source_column", "event_name"], observed=True)
            .agg(number_of_occurrences=("date", "count"), mean_daily_units=("total_units_sold", "mean"))
            .reset_index()
            .sort_values("number_of_occurrences", ascending=False)
        )
        display(eventNameSummary.head(20))
        saveTable(eventNameSummary, "sales_by_event_name.csv")
else:
    print("No event-name columns were found in the loaded calendar data; skipping event analysis.")

# If event types are available, see which ones show up often and how sales look on those days.
eventTypeColumns = [columnName for columnName in ["event_type_1", "event_type_2"] if columnName in dailySales.columns]
if eventTypeColumns:
    eventTypeRecords = []
    for eventColumn in eventTypeColumns:
        eventRows = dailySales.loc[dailySales[eventColumn].notna(), ["date", "total_units_sold", eventColumn]].copy()
        eventRows = eventRows.rename(columns={eventColumn: "event_type"})
        eventRows["event_source_column"] = eventColumn
        eventTypeRecords.append(eventRows)
    eventTypeFrame = pd.concat(eventTypeRecords, ignore_index=True)
    eventTypeSummary = (
        eventTypeFrame.groupby(["event_source_column", "event_type"], observed=True)
        .agg(number_of_occurrences=("date", "count"), mean_daily_units=("total_units_sold", "mean"))
        .reset_index().sort_values("number_of_occurrences", ascending=False)
    )
    display(eventTypeSummary)
    saveTable(eventTypeSummary, "sales_by_event_type.csv")

##### SNAP(food stamp) indicators by state

The calendar includes SNAP indicators for California (`snap_CA`), Texas (`snap_TX`) and Wisconsin (`snap_WI`). We'll compare state-level sales on active and inactive days. Any difference is an association, not proof that SNAP caused a change in sales.

In [ ]:
# SNAP is state-specific in this dataset, so compare each state with its own indicator.
snapColumnsByState = {"CA": "snap_CA", "TX": "snap_TX", "WI": "snap_WI"}
snapResultRows = []
stateDailyFrames = []

if "state_id" in salesData.columns:
    for stateCode, snapColumn in snapColumnsByState.items():
        if snapColumn not in calendarForEda.columns:
            continue
        stateMask = salesData["state_id"].astype(str).eq(stateCode)
        if not stateMask.any():
            continue
        stateSalesByDay = salesData.loc[stateMask, dayColumns].sum(axis=0).to_numpy(dtype=np.int64)
        stateDaily = pd.DataFrame({
            "d": dayColumns,
            "state_id": stateCode,
            "total_units_sold": stateSalesByDay,
        }).merge(
            calendarForEda[["d", "date", snapColumn]],
            on="d", how="left", validate="one_to_one"
        )
        stateDaily["snap_active"] = stateDaily[snapColumn].fillna(0).astype(int).eq(1)
        comparison = (
            stateDaily.groupby("snap_active", observed=True)
            .agg(
                number_of_days=("date", "count"),
                mean_daily_units=("total_units_sold", "mean"),
                median_daily_units=("total_units_sold", "median"),
            ).reset_index()
        )
        for _, comparisonRow in comparison.iterrows():
            snapResultRows.append({
                "state_id": stateCode,
                "snap_active": bool(comparisonRow["snap_active"]),
                "number_of_days": int(comparisonRow["number_of_days"]),
                "mean_daily_units": float(comparisonRow["mean_daily_units"]),
                "median_daily_units": float(comparisonRow["median_daily_units"]),
            })
        stateDailyFrames.append(stateDaily[["date", "state_id", "total_units_sold", "snap_active"]])

snapComparison = pd.DataFrame(snapResultRows)
if not snapComparison.empty:
    snapComparison["day_group"] = snapComparison["snap_active"].map({True: "SNAP indicator active", False: "SNAP indicator inactive"})
    display(snapComparison)
    saveTable(snapComparison, "snap_sales_comparison_by_state.csv")

    figure, axis = plt.subplots(figsize=(10, 5))
    pivotSnap = snapComparison.pivot(index="state_id", columns="day_group", values="mean_daily_units")
    pivotSnap.plot(kind="bar", ax=axis)
    axis.set_title("Mean state-level daily sales by SNAP indicator")
    axis.set_xlabel("State")
    axis.set_ylabel("Mean units sold per day")
    axis.tick_params(axis="x", rotation=0)
    axis.legend(title="Calendar indicator")
    saveCurrentFigure("eda_snap_indicator_by_state.png")
else:
    print("SNAP comparison skipped: expected state IDs or SNAP columns were not available.")

##### What do prices look like, and how do they line up with sales?

Prices are recorded weekly by item and store, while sales are daily. We'll inspect price variation, then aggregate a manageable sample of daily sales to the week and match it to the relevant price records. Price and sales moving together doesn't prove causation; when forecasting, we'll also need to use only prices that would have been known at prediction time.

In [ ]:
if "sell_price" in priceData.columns:
    priceDescription = priceData["sell_price"].describe(percentiles=[0.01, 0.10, 0.25, 0.50, 0.75, 0.90, 0.99]).to_frame("sell_price")
    display(priceDescription)
    saveTable(priceDescription, "sell_price_descriptive_statistics.csv")

    priceSample = priceData["sell_price"].dropna().sample(
        n=min(150_000, int(priceData["sell_price"].notna().sum())),
        random_state=randomSeed,
    )
    figure, axis = plt.subplots(figsize=(10, 5))
    axis.hist(priceSample, bins=60, edgecolor="black", linewidth=0.25)
    axis.set_title("Distribution of selling prices (random sample of price records)")
    axis.set_xlabel("Selling price")
    axis.set_ylabel("Number of sampled records")
    saveCurrentFigure("eda_sell_price_distribution.png")

    figure, axis = plt.subplots(figsize=(10, 5))
    positivePrices = priceSample[priceSample > 0]
    if not positivePrices.empty:
        axis.hist(np.log1p(positivePrices), bins=60, edgecolor="black", linewidth=0.25)
        axis.set_title("Log-transformed selling-price distribution (sample)")
        axis.set_xlabel("log(1 + selling price)")
        axis.set_ylabel("Number of sampled records")
        saveCurrentFigure("eda_sell_price_log_distribution.png")

    # Compare each product-store price range with its average price.
    priceVariation = (
        priceData.groupby(["store_id", "item_id"], observed=True)["sell_price"]
        .agg(number_of_price_weeks="count", minimum_price="min", maximum_price="max", mean_price="mean", std_price="std")
        .reset_index()
    )
    priceVariation["relative_price_range"] = np.divide(
        priceVariation["maximum_price"] - priceVariation["minimum_price"],
        priceVariation["mean_price"],
        out=np.full(len(priceVariation), np.nan, dtype=np.float64),
        where=priceVariation["mean_price"].to_numpy(dtype=np.float64) > 0,
    )
    display(priceVariation.describe(include="all").T)
    saveTable(priceVariation, "price_variation_by_item_store.csv")

    # Sample a few series across categories and add their daily sales up to the weekly price level.
    sampleParts = []
    if "cat_id" in salesData.columns:
        for _, categoryFrame in salesData.groupby("cat_id", observed=True, sort=False):
            sampleParts.append(categoryFrame.sample(
                n=min(20, len(categoryFrame)), random_state=randomSeed
            ))
    else:
        sampleParts = [salesData.sample(n=min(60, len(salesData)), random_state=randomSeed)]

    sampledSales = pd.concat(sampleParts, ignore_index=True).drop_duplicates(subset=["id"])
    sampleIdColumns = [columnName for columnName in ["id", "item_id", "store_id", "cat_id"] if columnName in sampledSales.columns]
    # Only reshape the sampled product-store series; melting the full table would be much heavier.
    sampledLong = sampledSales[sampleIdColumns + dayColumns].melt(
        id_vars=sampleIdColumns,
        value_vars=dayColumns,
        var_name="d",
        value_name="daily_units_sold",
    )
    for keyColumn in ["item_id", "store_id"]:
        sampledLong[keyColumn] = sampledLong[keyColumn].astype(str)
    calendarPriceKeys = calendarForEda[["d", "date", "wm_yr_wk"]].copy()
    calendarPriceKeys["d"] = calendarPriceKeys["d"].astype(str)
    sampledLong = sampledLong.merge(
        calendarPriceKeys, on="d", how="left", validate="many_to_one"
    )
    weeklySampleSales = (
        sampledLong.groupby(sampleIdColumns + ["wm_yr_wk"], observed=True, as_index=False)["daily_units_sold"]
        .sum().rename(columns={"daily_units_sold": "weekly_units_sold"})
    )
    pricesForMerge = priceData[["store_id", "item_id", "wm_yr_wk", "sell_price"]].copy()
    for keyColumn in ["item_id", "store_id"]:
        pricesForMerge[keyColumn] = pricesForMerge[keyColumn].astype(str)
    # Match sales aggregated to weeks with the price recorded for that same store, item and week.
    weeklyPriceDemand = weeklySampleSales.merge(
        pricesForMerge,
        on=["store_id", "item_id", "wm_yr_wk"],
        how="left",
        validate="many_to_one",
    )

    samplePriceCoverage = float(weeklyPriceDemand["sell_price"].notna().mean()) if len(weeklyPriceDemand) else np.nan
    print(f"Sampled product-store series: {sampledSales['id'].nunique():,}")
    print(f"Weekly sales observations with a matching price: {samplePriceCoverage:.2%}")
    saveTable(weeklyPriceDemand, "sampled_weekly_sales_and_prices.csv")

    # This scatter mixes products, so treat it as an overview. The correlations below are within each series.
    matchedPriceSales = weeklyPriceDemand.dropna(subset=["sell_price"]).sample(
        n=min(8000, int(weeklyPriceDemand["sell_price"].notna().sum())),
        random_state=randomSeed,
    )
    if not matchedPriceSales.empty:
        figure, axis = plt.subplots(figsize=(10, 6))
        axis.scatter(matchedPriceSales["sell_price"], matchedPriceSales["weekly_units_sold"], alpha=0.25, s=12)
        axis.set_title("Weekly selling price versus weekly sales for sampled series")
        axis.set_xlabel("Selling price")
        axis.set_ylabel("Units sold per week")
        saveCurrentFigure("eda_weekly_price_vs_sales_sample.png")

    # Correlate price and sales within each series, rather than mixing all products together.
    perSeriesPriceCorrelations = []
    for seriesId, seriesGroup in weeklyPriceDemand.dropna(subset=["sell_price"]).groupby("id", observed=True):
        if len(seriesGroup) >= 8 and seriesGroup["sell_price"].nunique() > 1 and seriesGroup["weekly_units_sold"].nunique() > 1:
            perSeriesPriceCorrelations.append({
                "id": str(seriesId),
                "number_of_weeks": len(seriesGroup),
                "price_sales_correlation": seriesGroup["sell_price"].corr(seriesGroup["weekly_units_sold"]),
            })
    priceSalesCorrelations = pd.DataFrame(perSeriesPriceCorrelations)
    if not priceSalesCorrelations.empty:
        display(priceSalesCorrelations["price_sales_correlation"].describe().to_frame("within_series_price_sales_correlation"))
        figure, axis = plt.subplots(figsize=(10, 5))
        axis.hist(priceSalesCorrelations["price_sales_correlation"].dropna(), bins=25, edgecolor="black", linewidth=0.25)
        axis.set_title("Within-series correlations between weekly price and weekly sales")
        axis.set_xlabel("Correlation for one product-store series")
        axis.set_ylabel("Number of sampled series")
        saveCurrentFigure("eda_within_series_price_sales_correlations.png")
        saveTable(priceSalesCorrelations, "sampled_price_sales_correlations.csv")
    else:
        print("Too few sampled series had enough price variation for within-series correlations.")
else:
    print("Price analysis skipped because sell_price was not found in the price table.")

##### Pulling the EDA together

This section gathers key statistics in one place. We note what we observed, how consistent the pattern was, and what it suggests we should test in the model. The EDA can guide the experiment, but only evaluation can tell us whether a forecasting approach performs better.

In [ ]:
# Pull the headline numbers together so we can refer to them when writing up the EDA.
overallMeanDailySales = float(dailySales["total_units_sold"].mean())
overallMedianDailySales = float(dailySales["total_units_sold"].median())
medianZeroShare = float(seriesSummary["zero_day_share"].median())
meanZeroShare = float(seriesSummary["zero_day_share"].mean())
medianCvAll = float(seriesSummary["cv_all_days"].median()) if seriesSummary["cv_all_days"].notna().any() else np.nan
medianAdi = float(seriesSummary["average_demand_interval"].median()) if seriesSummary["average_demand_interval"].notna().any() else np.nan

edaKeyStatistics = pd.DataFrame([
    ("Number of product-store series", len(seriesSummary)),
    ("Number of observed sales days per series", len(dayColumns)),
    ("First observed sales date", str(dailySales["date"].min().date())),
    ("Last observed sales date", str(dailySales["date"].max().date())),
    ("Total units recorded over the dataset", int(dailySales["total_units_sold"].sum())),
    ("Mean total units sold per day across the dataset", overallMeanDailySales),
    ("Median total units sold per day across the dataset", overallMedianDailySales),
    ("Zero-sales cells as a share of all series-day observations", zeroShareOverall),
    ("Mean zero-day share across product-store series", meanZeroShare),
    ("Median zero-day share across product-store series", medianZeroShare),
    ("Median CV across all days", medianCvAll),
    ("Median average demand interval (days between positive-demand frequency proxy)", medianAdi),
], columns=["metric", "value"])
display(edaKeyStatistics)
saveTable(edaKeyStatistics, "eda_key_statistics.csv")

print("\nEDA interpretation checklist")
print("1. Sales over time: note long-term trends, abrupt changes, and recurring peaks/troughs.")
print("2. Cross-sectional differences: identify stores/categories with distinct demand levels.")
print("3. Sparsity: record the share of series with frequent zero-sales days.")
print("4. Volatility: compare all-day CV with positive-demand CV and ADI.")
print("5. Seasonality: identify weekday/month patterns that warrant seasonal baselines or calendar features.")
print("6. Events/SNAP: describe observed associations without claiming causation from these comparisons alone.")
print("7. Prices: record price coverage and within-series associations; avoid causal claims.")
print("8. Modelling decision: document which features and model comparisons the observations justify.")
print(f"EDA tables saved in: {tablesDir}")
print(f"EDA figures saved in: {figuresDir}")

##### Pooling more series: all four California stores' FOODS items

Every prior branch trained on CA_1/FOODS alone (1,437 series). The single biggest win so far came from giving the model more *relevant* recent history (short-horizon lags); the next candidate lever is giving it more *relevant* training data outright. This notebook pools all four California stores' FOODS series (CA_1, CA_2, CA_3, CA_4 -- 5,748 series, 4x the data) into one model, keeping `store_id` as a feature so it can still specialize per store rather than averaging across them.

Staying within FOODS and within one state keeps two things fixed that would otherwise confound the comparison: the demand-sparsity profile (FOODS has the lowest zero-sales share) and the SNAP indicator (CA stores all key off `snap_CA`). Everything else -- scope definition, short-horizon lag features (1/7/14/21-day lags, 3/7/14-day rolling stats), Tweedie objective, chronological splits -- is identical to `LightGBM_BaseLocal.ipynb`, the current best model.

**The key methodological point**: the EDA already showed CA_1-CA_4 don't sell at the same volume (CA_3 is the highest-selling store in the whole dataset; CA_4 is comparatively low). A pooled model's overall RMSE on all four stores isn't directly comparable to CA_1-only's RMSE -- that's the same scale-confound we ran into comparing the old mixed-category sample to CA_1/FOODS. The real test of "did pooling help" is **narrower**: take the pooled model's predictions for CA_1 rows only, and compare that against the standalone CA_1-only model's recorded numbers (development validation RMSE 2.356 / MAE 1.308; later backtest RMSE 2.377 / MAE 1.377) on the exact same CA_1 rows and windows. Same evaluation population, only the training population differs.

##### How different are the four stores? (why the scale caveat matters)

In [ ]:
storeComparisonRows = []
for storeId in ["CA_1", "CA_2", "CA_3", "CA_4"]:
    storeMask = (salesData["store_id"] == storeId) & (salesData["cat_id"] == "FOODS")
    storeValues = salesData.loc[storeMask, dayColumns].to_numpy(dtype="float64")
    storeComparisonRows.append((
        storeId, int(storeMask.sum()), float(storeValues.mean()), float((storeValues == 0).mean())
    ))
storeComparison = pd.DataFrame(storeComparisonRows, columns=["store_id", "series", "mean_daily_sales", "zero_share"])
display(storeComparison.round(3))


##### Scoping to CA_1-CA_4 / FOODS (pooled)

In [ ]:
import gc

SCOPE_STATE_ID = "CA"
SCOPE_STORE_IDS = ["CA_1", "CA_2", "CA_3", "CA_4"]
SCOPE_CATEGORY_ID = "FOODS"

evaluationFilePath = rawDataDir / "sales_train_evaluation.csv"
if not evaluationFilePath.is_file():
    raise FileNotFoundError(f"Required training file was not found: {evaluationFilePath}")

evaluationHeader = pd.read_csv(evaluationFilePath, nrows=0)
evaluationDayColumns = [
    columnName for columnName in evaluationHeader.columns
    if re.fullmatch(r"d_\d+", str(columnName))
]
evaluationDataTypes = {columnName: "category" for columnName in identifierColumns}
evaluationDataTypes.update({columnName: "uint16" for columnName in evaluationDayColumns})
evaluationSales = pd.read_csv(evaluationFilePath, dtype=evaluationDataTypes, low_memory=False)

if len(dayColumns) != 1913 or len(evaluationDayColumns) != 1941:
    raise ValueError("Expected the validation file through d_1913 and the evaluation file through d_1941.")

scopedEvaluationSales = evaluationSales.loc[
    (evaluationSales["state_id"] == SCOPE_STATE_ID) & (evaluationSales["cat_id"] == SCOPE_CATEGORY_ID)
].reset_index(drop=True)
del evaluationSales
gc.collect()

scopedValidationSales = salesData.loc[
    (salesData["state_id"] == SCOPE_STATE_ID) & (salesData["cat_id"] == SCOPE_CATEGORY_ID)
].reset_index(drop=True)

if scopedEvaluationSales.empty:
    raise ValueError(f"No series found for state_id={SCOPE_STATE_ID!r} and cat_id={SCOPE_CATEGORY_ID!r}.")

print(f"Scoped series ({SCOPE_STATE_ID} / {SCOPE_CATEGORY_ID}, {len(SCOPE_STORE_IDS)} stores): {len(scopedEvaluationSales):,}")

scopedKey = ["item_id", "store_id"]
if scopedEvaluationSales.duplicated(scopedKey).any() or scopedValidationSales.duplicated(scopedKey).any():
    raise ValueError("Scoped sales contain duplicate item-store series.")

sortedEvaluation = scopedEvaluationSales.sort_values(scopedKey, kind="stable").reset_index(drop=True)
sortedValidation = scopedValidationSales.sort_values(scopedKey, kind="stable").reset_index(drop=True)
if not sortedEvaluation[scopedKey].astype("string").equals(sortedValidation[scopedKey].astype("string")):
    raise ValueError("Validation and evaluation files do not contain the same scoped item-store series.")
if not np.array_equal(sortedValidation[dayColumns].to_numpy(), sortedEvaluation[dayColumns].to_numpy()):
    raise ValueError("Validation and evaluation sales differ over their overlapping days.")

scopedSales = sortedEvaluation
print("Validation/evaluation overlap check passed; scoped series are consistent across both files.")


##### Calendar, SNAP and price features for the pooled subset

In [ ]:
calendarFeatures = calendarData.copy()
calendarFeatures["d_num"] = calendarFeatures["d"].astype(str).str.removeprefix("d_").astype("int32")
calendarFeatures["weekday_num"] = calendarFeatures["date"].dt.dayofweek.astype("int8")
calendarFeatures["month_num"] = calendarFeatures["date"].dt.month.astype("int8")
calendarFeatures["year_num"] = calendarFeatures["date"].dt.year.astype("int16")
calendarFeatures["day_of_month"] = calendarFeatures["date"].dt.day.astype("int8")
calendarFeatures["is_weekend"] = calendarFeatures["weekday_num"].ge(5).astype("int8")
for eventColumn in ["event_name_1", "event_type_1", "event_name_2", "event_type_2"]:
    calendarFeatures[eventColumn] = calendarFeatures[eventColumn].astype("category")

weekNumbers = sorted(calendarFeatures["wm_yr_wk"].dropna().unique().tolist())
weekIndex = {int(week): index for index, week in enumerate(weekNumbers)}
calendarFeatures["week_index"] = calendarFeatures["wm_yr_wk"].map(weekIndex).astype("int16")

calendarFeatureColumns = [
    "d_num", "date", "wm_yr_wk", "week_index", "weekday_num", "month_num", "year_num",
    "day_of_month", "is_weekend", "event_name_1", "event_type_1",
    "event_name_2", "event_type_2", "snap_CA",
]
# Every scoped store is in California, so snap_CA applies uniformly -- same trick as the single-store notebooks.
calendarFeatures = calendarFeatures[calendarFeatureColumns].rename(columns={"snap_CA": "snap"})

LATER_BACKTEST_END = 1941
if int(calendarFeatures["d_num"].max()) < LATER_BACKTEST_END:
    raise ValueError("Calendar does not cover the later backtest window.")

scopedPrices = priceData.loc[priceData["store_id"].astype(str).isin(SCOPE_STORE_IDS)].copy()
scopedPrices["store_id"] = scopedPrices["store_id"].astype(str)
scopedPrices["item_id"] = scopedPrices["item_id"].astype(str)

priceKeyColumns = ["store_id", "item_id", "wm_yr_wk"]
if scopedPrices.duplicated(priceKeyColumns).any():
    raise ValueError("Scoped price keys contain duplicates.")

releaseWeeks = (
    scopedPrices.groupby(["store_id", "item_id"], as_index=False)["wm_yr_wk"]
    .min()
    .rename(columns={"wm_yr_wk": "first_price_week"})
)
releaseWeeks["release_week_index"] = releaseWeeks["first_price_week"].map(weekIndex)

print(f"Calendar days available: {len(calendarFeatures):,}")
print(f"Scoped price rows: {len(scopedPrices):,}")

# Free the full 30,490-series EDA tables now that their scoped derivatives exist -- holding them through the ~10.7M-row pooled feature build is what likely drove the earlier OOM.
del salesData, calendarData, priceData
gc.collect()


##### Lag and rolling features on the wide matrix (no `pd.melt`), same short-horizon design as the current best model

In [ ]:
HORIZON_DAYS = 1
LAG_DAYS = [1, 7, 14, 21]
ROLLING_WINDOWS = [3, 7, 14]
ROLLING_DELAY = HORIZON_DAYS
FEATURE_START_DAY = max(LAG_DAYS) + 1
DEVELOPMENT_TRAIN_END = 1885
DEVELOPMENT_VALIDATION_START, DEVELOPMENT_VALIDATION_END = 1886, 1913
LATER_BACKTEST_START = 1914


def computeRollingStats(salesMatrix: np.ndarray, targetDayNumbers: np.ndarray, window: int) -> tuple[np.ndarray, np.ndarray]:
    """Mean/std over `window` days ending ROLLING_DELAY days before each target day, via cumulative sums."""
    prefixSum = np.concatenate(
        [np.zeros((salesMatrix.shape[0], 1)), np.cumsum(salesMatrix, axis=1, dtype=np.float64)], axis=1
    )
    prefixSquareSum = np.concatenate(
        [np.zeros((salesMatrix.shape[0], 1)), np.cumsum(salesMatrix.astype(np.float64) ** 2, axis=1)], axis=1
    )
    targetIndexes = targetDayNumbers.astype(np.int64) - 1
    windowEnd = targetIndexes - ROLLING_DELAY + 1
    windowStart = windowEnd - window
    if windowStart.min() < 0 or windowEnd.max() > salesMatrix.shape[1]:
        raise ValueError("Not enough historical days for the requested rolling window.")
    windowSum = prefixSum[:, windowEnd] - prefixSum[:, windowStart]
    windowSquareSum = prefixSquareSum[:, windowEnd] - prefixSquareSum[:, windowStart]
    windowMean = windowSum / window
    windowVariance = np.maximum((windowSquareSum - (windowSum ** 2) / window) / (window - 1), 0.0)
    return windowMean.astype("float32"), np.sqrt(windowVariance).astype("float32")


salesMatrix = scopedSales[evaluationDayColumns].to_numpy(dtype="float32", copy=True)
seriesCount = len(scopedSales)
targetDayNumbers = np.arange(FEATURE_START_DAY, LATER_BACKTEST_END + 1, dtype=np.int32)

featureFrameData: dict[str, object] = {}
for identityColumn in identifierColumns:
    featureFrameData[identityColumn] = np.tile(
        scopedSales[identityColumn].astype(str).to_numpy(dtype=object), len(targetDayNumbers)
    )
featureFrameData["d_num"] = np.repeat(targetDayNumbers, seriesCount).astype("int32")

for lag in LAG_DAYS:
    sourceIndexes = targetDayNumbers.astype(np.int64) - 1 - lag
    if sourceIndexes.min() < 0:
        raise ValueError(f"Not enough history for lag {lag}.")
    featureFrameData[f"sales_lag_{lag}"] = salesMatrix[:, sourceIndexes].T.reshape(-1)

for window in ROLLING_WINDOWS:
    meanValues, stdValues = computeRollingStats(salesMatrix, targetDayNumbers, window)
    featureFrameData[f"rolling_mean_{window}"] = meanValues.T.reshape(-1)
    featureFrameData[f"rolling_std_{window}"] = stdValues.T.reshape(-1)

featureFrameData["sales"] = salesMatrix[:, targetDayNumbers - 1].T.reshape(-1)

featureFrame = pd.DataFrame(featureFrameData)
print(f"Scoped long-format rows: {len(featureFrame):,} (built without pd.melt)")
del salesMatrix, featureFrameData
gc.collect()


##### Join calendar, SNAP and price features, then split chronologically

In [ ]:
featureFrame = featureFrame.merge(calendarFeatures, on="d_num", how="left", validate="many_to_one")
if featureFrame["wm_yr_wk"].isna().any():
    raise ValueError("Calendar join left unmatched days.")

featureFrame = featureFrame.merge(
    scopedPrices[["store_id", "item_id", "wm_yr_wk", "sell_price"]],
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left",
    validate="many_to_one",
)
featureFrame["price_missing"] = featureFrame["sell_price"].isna().astype("int8")

featureFrame = featureFrame.merge(
    releaseWeeks[["store_id", "item_id", "release_week_index"]],
    on=["store_id", "item_id"],
    how="left",
    validate="many_to_one",
)
featureFrame["weeks_since_first_price"] = (
    featureFrame["week_index"].astype("float32") - featureFrame["release_week_index"].astype("float32")
)
featureFrame = featureFrame.drop(columns=["release_week_index", "week_index"])

categoricalFeatureColumns = identifierColumns + ["event_name_1", "event_type_1", "event_name_2", "event_type_2"]
for categoricalColumn in categoricalFeatureColumns:
    featureFrame[categoricalColumn] = featureFrame[categoricalColumn].astype("category")

dayNumber = featureFrame["d_num"]
developmentTrain = featureFrame.loc[dayNumber.between(FEATURE_START_DAY, DEVELOPMENT_TRAIN_END)].reset_index(drop=True)
developmentValidation = featureFrame.loc[
    dayNumber.between(DEVELOPMENT_VALIDATION_START, DEVELOPMENT_VALIDATION_END)
].reset_index(drop=True)
laterBacktest = featureFrame.loc[dayNumber.between(LATER_BACKTEST_START, LATER_BACKTEST_END)].reset_index(drop=True)
del featureFrame
gc.collect()

print(f"Development train rows: {len(developmentTrain):,} (d_{FEATURE_START_DAY}-d_{DEVELOPMENT_TRAIN_END})")
print(f"Development validation rows: {len(developmentValidation):,} (d_{DEVELOPMENT_VALIDATION_START}-d_{DEVELOPMENT_VALIDATION_END})")
print(f"Later backtest rows: {len(laterBacktest):,} (d_{LATER_BACKTEST_START}-d_{LATER_BACKTEST_END})")


##### Fit a single pooled LightGBM model (Tweedie objective)

In [ ]:
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

featureColumns = [column for column in developmentTrain.columns if column not in {"sales", "date"}]
xTrain, yTrain = developmentTrain[featureColumns], developmentTrain["sales"]
xValidation, yValidation = developmentValidation[featureColumns], developmentValidation["sales"]

print(f"Training rows: {len(xTrain):,}")
print(f"Validation rows: {len(xValidation):,}")
print(f"Features ({len(featureColumns)}): {featureColumns}")


In [ ]:
pooledModel = lgb.LGBMRegressor(
    objective="tweedie",
    tweedie_variance_power=1.1,
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    n_jobs=-1,
    random_state=randomSeed,
    verbosity=-1,
)
pooledModel.fit(
    xTrain,
    yTrain,
    eval_set=[(xValidation, yValidation)],
    eval_metric="rmse",
    callbacks=[lgb.early_stopping(30), lgb.log_evaluation(25)],
)

pooledPredictions = pooledModel.predict(xValidation, num_iteration=pooledModel.best_iteration_)
pooledBaseline = xValidation["sales_lag_1"].to_numpy()

pooledMetrics = pd.DataFrame(
    [
        ("Pooled LightGBM, Tweedie (CA_1-CA_4 / FOODS, all stores)", root_mean_squared_error(yValidation, pooledPredictions), mean_absolute_error(yValidation, pooledPredictions)),
        ("Naive lag-1 baseline (pooled)", root_mean_squared_error(yValidation, pooledBaseline), mean_absolute_error(yValidation, pooledBaseline)),
    ],
    columns=["model", "RMSE", "MAE"],
)
display(pooledMetrics.round(4))


##### Per-store breakdown (pooled model's predictions, split out by store)

In [ ]:
perStoreRows = []
for storeId in SCOPE_STORE_IDS:
    storeMask = (xValidation["store_id"] == storeId).to_numpy()
    perStoreRows.append((
        storeId,
        int(storeMask.sum()),
        root_mean_squared_error(yValidation.loc[storeMask], pooledPredictions[storeMask]),
        mean_absolute_error(yValidation.loc[storeMask], pooledPredictions[storeMask]),
    ))
perStoreMetrics = pd.DataFrame(perStoreRows, columns=["store_id", "validation_rows", "RMSE", "MAE"])
display(perStoreMetrics.round(4))


##### The real test: does pooling help CA_1 specifically?

Same evaluation rows (CA_1, development validation window) as `LightGBM_BaseLocal.ipynb`'s recorded result -- only the training population differs (CA_1 alone vs. CA_1-CA_4 pooled).

In [ ]:
ca1ValidationMask = (xValidation["store_id"] == "CA_1").to_numpy()
ca1PooledRmse = root_mean_squared_error(yValidation.loc[ca1ValidationMask], pooledPredictions[ca1ValidationMask])
ca1PooledMae = mean_absolute_error(yValidation.loc[ca1ValidationMask], pooledPredictions[ca1ValidationMask])

ca1Comparison = pd.DataFrame(
    [
        ("CA_1-only model (from LightGBM_BaseLocal.ipynb)", 2.356, 1.308),
        ("Pooled model (CA_1-CA_4), evaluated on CA_1 rows only", ca1PooledRmse, ca1PooledMae),
    ],
    columns=["model", "RMSE", "MAE"],
)
display(ca1Comparison.round(4))


##### Later backtest (d_1914-d_1941): refit on all labelled history and re-check

In [ ]:
combinedTrain = pd.concat([developmentTrain, developmentValidation], ignore_index=True)
xCombinedTrain, yCombinedTrain = combinedTrain[featureColumns], combinedTrain["sales"]
xLaterBacktest, yLaterBacktest = laterBacktest[featureColumns], laterBacktest["sales"]

backtestModel = lgb.LGBMRegressor(
    objective="tweedie",
    tweedie_variance_power=1.1,
    n_estimators=pooledModel.best_iteration_ or pooledModel.n_estimators,
    learning_rate=0.05,
    num_leaves=31,
    n_jobs=-1,
    random_state=randomSeed,
    verbosity=-1,
)
backtestModel.fit(xCombinedTrain, yCombinedTrain)
backtestPredictions = backtestModel.predict(xLaterBacktest)
backtestBaseline = xLaterBacktest["sales_lag_1"].to_numpy()

backtestMetrics = pd.DataFrame(
    [
        ("Pooled LightGBM later backtest (all CA/FOODS stores)", root_mean_squared_error(yLaterBacktest, backtestPredictions), mean_absolute_error(yLaterBacktest, backtestPredictions)),
        ("Naive lag-1 baseline later backtest (pooled)", root_mean_squared_error(yLaterBacktest, backtestBaseline), mean_absolute_error(yLaterBacktest, backtestBaseline)),
    ],
    columns=["model", "RMSE", "MAE"],
)
display(backtestMetrics.round(4))

ca1BacktestMask = (xLaterBacktest["store_id"] == "CA_1").to_numpy()
ca1PooledBacktestRmse = root_mean_squared_error(yLaterBacktest.loc[ca1BacktestMask], backtestPredictions[ca1BacktestMask])
ca1PooledBacktestMae = mean_absolute_error(yLaterBacktest.loc[ca1BacktestMask], backtestPredictions[ca1BacktestMask])

ca1BacktestComparison = pd.DataFrame(
    [
        ("CA_1-only model later backtest (from LightGBM_BaseLocal.ipynb)", 2.377, 1.377),
        ("Pooled model later backtest, evaluated on CA_1 rows only", ca1PooledBacktestRmse, ca1PooledBacktestMae),
    ],
    columns=["model", "RMSE", "MAE"],
)
display(ca1BacktestComparison.round(4))


##### Findings

**Pooled model vs. the naive baseline.** One Tweedie LightGBM trained on all 5,748 CA_1-CA_4 FOODS series beats the naive lag-1 baseline by about 27% on RMSE and 20% on MAE, and the gap holds on the later backtest.

| Window | Model | RMSE | MAE |
|---|---|---|---|
| Development validation (d_1886-d_1913) | Pooled LightGBM | 2.283 | 1.279 |
| Development validation (d_1886-d_1913) | Naive lag-1 baseline | 3.116 | 1.597 |
| Later backtest (d_1914-d_1941) | Pooled LightGBM | 2.285 | 1.319 |
| Later backtest (d_1914-d_1941) | Naive lag-1 baseline | 3.088 | 1.661 |

**Per store.** Error follows sales volume. CA_3 sells the most per day (2.73) and has the highest RMSE (2.831). CA_4 sells the least (1.03) and has the lowest (1.549). CA_1 and CA_2 sit in between at 2.370 and 2.194.

**Does pooling help CA_1?** Not meaningfully. Evaluated on the same CA_1 rows as `LightGBM_BaseLocal.ipynb`:

| Window | CA_1-only RMSE / MAE | Pooled RMSE / MAE |
|---|---|---|
| Development validation | 2.356 / 1.308 | 2.370 / 1.305 |
| Later backtest | 2.377 / 1.377 | 2.377 / 1.371 |

RMSE is flat or slightly worse, and MAE is better by only 0.003-0.006, which is within noise. Four times the training data (about 10.7M rows instead of roughly 2.7M) bought no real accuracy on CA_1.

**Likely reasons.** The short-horizon lag and rolling features already carry most of the signal, and the stores differ in scale and zero-sales share (52-67%), which may dilute what the other stores add.

**Takeaway.** Keep the CA_1-only model if CA_1 is the target. Use the pooled model if one model for all four stores is preferred: it is competitive on each store and cheaper to maintain than four separate models, though it is slower to train.